# Decision Tree & Random Forest — No-Framework Implementation

Building both Decision Tree and Random Forest from scratch using only NumPy. Most algorithmically complex from-scratch build in the project — involves a recursive data structure (the tree itself).

**Part 1: Decision Tree** (baseline — demonstrates overfitting)
- Recursive tree building with Gini impurity split criterion
- Weighted impurity for class imbalance (88.7/11.3 ratio)
- Depth analysis sweep to find optimal `max_depth`

**Part 2: Random Forest** (main event — fixes overfitting via bagging)
- Bootstrap sampling + random feature subsets (`sqrt(n_features)`)
- Majority vote across 100 trees, averaged probabilities
- Manual OOB score computation (our showcase)

**Showcase**: Gini vs Entropy comparison (Part 1) + Manual OOB score (Part 2) — demonstrating core math behind split criteria and out-of-bag validation.

## What We Build From Scratch
- **Gini impurity**: `1 - sum(p_k^2)` — probability of misclassification
- **Entropy**: `-sum(p_k * log(p_k))` — information content
- **Information gain**: parent impurity minus weighted child impurity
- **Recursive tree building**: greedy best-split search at each node
- **Bootstrap aggregating (bagging)**: sampling with replacement + ensemble vote
- **OOB score**: validation using out-of-bag samples per tree

In [1]:
# Step 1: Imports, config, and data loading
import numpy as np
import sys
import os

# Add project root to path for shared utilities
sys.path.insert(0, os.path.abspath('../..'))

from utils.data_loader import load_processed_data
from utils.metrics import evaluate_classifier, print_metrics
from utils.performance import track_performance, track_inference, get_model_size
from utils.visualization import (plot_confusion_matrix, plot_roc_curve,
                                  plot_feature_importance, plot_calibration_curve,
                                  plot_tree_depth_analysis, plot_forest_convergence)
from utils.results import save_results, add_result, print_comparison, build_results_dict
from utils.tree_utils import compute_feature_importance, flatten_tree, predict_batch

# Configuration
RANDOM_STATE = 113
FRAMEWORK = 'No-Framework'
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# DT depth sweep values (same as sklearn for fair comparison)
DEPTH_VALUES = [2, 3, 5, 7, 10, 15, 20, None]

# RF configuration
N_ESTIMATORS = 100
MAX_FEATURES = 'sqrt'  # sqrt(19) ≈ 4 random features per split

# Set random seed for reproducibility
np.random.seed(RANDOM_STATE)

# Load preprocessed data
X_train, X_test, y_train, y_test, metadata = load_processed_data('decision_tree')
feature_names = metadata['feature_names']
n_features = metadata['n_features']

print("=" * 60)
print(f"DECISION TREE & RANDOM FOREST — {FRAMEWORK}")
print("=" * 60)
print(f"\nDataset: {metadata['dataset']}")
print(f"Features: {n_features} ({len(metadata['categorical_features'])} categorical, "
      f"{len(metadata['continuous_features'])} continuous)")
print(f"Training: {X_train.shape[0]:,} samples")
print(f"Test:     {X_test.shape[0]:,} samples")
print(f"Classes:  {metadata['class_names']}")
print(f"Train class dist: 0={np.sum(y_train==0):,}, 1={np.sum(y_train==1):,} "
      f"({np.mean(y_train==1)*100:.1f}% positive)")
print(f"\nDropped: {list(metadata['dropped_features'].keys())} (data leakage)")
print(f"Scaling: {metadata['scaling']}")

DECISION TREE & RANDOM FOREST — No-Framework

Dataset: Bank Marketing (UCI)
Features: 19 (10 categorical, 9 continuous)
Training: 32,950 samples
Test:     8,238 samples
Classes:  ['no', 'yes']
Train class dist: 0=29,238, 1=3,712 (11.3% positive)

Dropped: ['duration'] (data leakage)
Scaling: None (trees are scale-invariant)


In [2]:
# Step 2: Split criterion functions
def compute_sample_weights(y):
    """
    Compute balanced class weights (matches sklearn's class_weight='balanced').

    Each sample gets weight = n_samples / (n_classes * n_samples_in_class).
    This makes the minority class (11.3% positive) contribute equally to
    impurity as the majority class, preventing the tree from always
    predicting 'no'.

    Args:
        y: Labels array (n_samples,)

    Returns:
        weights: Per-sample weight array (n_samples,)
    """
    classes = np.unique(y)
    n_samples = len(y)
    n_classes = len(classes)

    weights = np.zeros(n_samples)
    for cls in classes:
        mask = y == cls
        n_cls = mask.sum()
        weights[mask] = n_samples / (n_classes * n_cls)

    return weights


def gini_impurity(y, sample_weight=None):
    """
    Gini impurity: probability of misclassifying a randomly chosen sample.

    Gini = 1 - sum(p_k^2), where p_k is the (weighted) proportion of class k.
    - Gini = 0.0: pure node (all samples same class)
    - Gini = 0.5: maximum impurity for binary (50/50 split)

    Args:
        y: Labels at this node (n_samples,)
        sample_weight: Per-sample weights (n_samples,). None = uniform.

    Returns:
        float: Gini impurity in [0, 0.5] for binary classification
    """
    if len(y) == 0:
        return 0.0

    classes = np.unique(y)

    if sample_weight is None:
        proportions = np.array([np.mean(y == c) for c in classes])
    else:
        total_weight = sample_weight.sum()
        if total_weight == 0:
            return 0.0
        proportions = np.array([
            sample_weight[y == c].sum() / total_weight for c in classes
        ])

    return 1.0 - np.sum(proportions ** 2)


def entropy_impurity(y, sample_weight=None):
    """
    Shannon entropy: information content of the class distribution.

    Entropy = -sum(p_k * log2(p_k)), where p_k is the (weighted) proportion.
    - Entropy = 0.0: pure node (all same class)
    - Entropy = 1.0: maximum for binary (50/50 split)

    Args:
        y: Labels at this node (n_samples,)
        sample_weight: Per-sample weights (n_samples,). None = uniform.

    Returns:
        float: Entropy >= 0
    """
    if len(y) == 0:
        return 0.0

    classes = np.unique(y)

    if sample_weight is None:
        proportions = np.array([np.mean(y == c) for c in classes])
    else:
        total_weight = sample_weight.sum()
        if total_weight == 0:
            return 0.0
        proportions = np.array([
            sample_weight[y == c].sum() / total_weight for c in classes
        ])

    # Filter out zeros to avoid log2(0)
    proportions = proportions[proportions > 0]
    return -np.sum(proportions * np.log2(proportions))


def information_gain(y_parent, y_left, y_right, criterion='gini',
                     sw_parent=None, sw_left=None, sw_right=None):
    """
    Information gain from splitting parent into left and right children.

    IG = impurity(parent) - weighted_average(impurity(children))
    Higher IG = split separates classes more effectively.

    Args:
        y_parent, y_left, y_right: Labels for parent and children
        criterion: 'gini' or 'entropy'
        sw_parent, sw_left, sw_right: Sample weights (None = uniform)

    Returns:
        float: Information gain >= 0
    """
    imp_func = gini_impurity if criterion == 'gini' else entropy_impurity

    parent_imp = imp_func(y_parent, sw_parent)

    # Weight children by their (weighted) sample count
    if sw_parent is None:
        n = len(y_parent)
        w_left = len(y_left) / n
        w_right = len(y_right) / n
    else:
        total = sw_parent.sum()
        w_left = sw_left.sum() / total
        w_right = sw_right.sum() / total

    child_imp = (w_left * imp_func(y_left, sw_left) +
                 w_right * imp_func(y_right, sw_right))

    return parent_imp - child_imp

In [3]:
# Test all functions
sample_weights = compute_sample_weights(y_train)

print("=" * 60)
print("[1/4] SAMPLE WEIGHTS (Balanced Class Weighting)")
print("=" * 60)
print(f"  Class 0 (no):  weight = {sample_weights[y_train == 0][0]:.4f} "
      f"({np.sum(y_train == 0):,} samples)")
print(f"  Class 1 (yes): weight = {sample_weights[y_train == 1][0]:.4f} "
      f"({np.sum(y_train == 1):,} samples)")
print(f"  Effective ratio: 1:{sample_weights[y_train == 1][0] / sample_weights[y_train == 0][0]:.1f} "
      f"(minority upweighted)")

print(f"\n{'=' * 60}")
print("[2/4] GINI IMPURITY")
print("=" * 60)
pure = np.array([0, 0, 0, 0, 0])
balanced = np.array([0, 0, 1, 1])
print(f"  Pure node [0,0,0,0,0]:       Gini = {gini_impurity(pure):.4f}")
print(f"  Balanced [0,0,1,1]:          Gini = {gini_impurity(balanced):.4f}")
print(f"  Training data (unweighted):  Gini = {gini_impurity(y_train):.4f}")
print(f"  Training data (weighted):    Gini = {gini_impurity(y_train, sample_weights):.4f}")

print(f"\n{'=' * 60}")
print("[3/4] ENTROPY")
print("=" * 60)
print(f"  Pure node [0,0,0,0,0]:       Entropy = {entropy_impurity(pure):.4f}")
print(f"  Balanced [0,0,1,1]:          Entropy = {entropy_impurity(balanced):.4f}")
print(f"  Training data (unweighted):  Entropy = {entropy_impurity(y_train):.4f}")
print(f"  Training data (weighted):    Entropy = {entropy_impurity(y_train, sample_weights):.4f}")

print(f"\n{'=' * 60}")
print("[4/4] INFORMATION GAIN (Sample Split)")
print("=" * 60)
feat_idx = 0
threshold = np.median(X_train[:, feat_idx])
left_mask = X_train[:, feat_idx] <= threshold
right_mask = ~left_mask

ig_gini = information_gain(
    y_train, y_train[left_mask], y_train[right_mask], criterion='gini',
    sw_parent=sample_weights, sw_left=sample_weights[left_mask],
    sw_right=sample_weights[right_mask]
)
ig_entropy = information_gain(
    y_train, y_train[left_mask], y_train[right_mask], criterion='entropy',
    sw_parent=sample_weights, sw_left=sample_weights[left_mask],
    sw_right=sample_weights[right_mask]
)
print(f"  Split: {feature_names[feat_idx]} <= {threshold:.2f}")
print(f"  Left: {left_mask.sum():,} samples, Right: {right_mask.sum():,} samples")
print(f"  IG (Gini):    {ig_gini:.6f}")
print(f"  IG (Entropy): {ig_entropy:.6f}")

[1/4] SAMPLE WEIGHTS (Balanced Class Weighting)
  Class 0 (no):  weight = 0.5635 (29,238 samples)
  Class 1 (yes): weight = 4.4383 (3,712 samples)
  Effective ratio: 1:7.9 (minority upweighted)

[2/4] GINI IMPURITY
  Pure node [0,0,0,0,0]:       Gini = 0.0000
  Balanced [0,0,1,1]:          Gini = 0.5000
  Training data (unweighted):  Gini = 0.1999
  Training data (weighted):    Gini = 0.5000

[3/4] ENTROPY
  Pure node [0,0,0,0,0]:       Entropy = -0.0000
  Balanced [0,0,1,1]:          Entropy = 1.0000
  Training data (unweighted):  Entropy = 0.5079
  Training data (weighted):    Entropy = 1.0000

[4/4] INFORMATION GAIN (Sample Split)
  Split: age <= 38.00
  Left: 16,972 samples, Right: 15,978 samples
  IG (Gini):    0.000510
  IG (Entropy): 0.000736


In [6]:
# Step 3: Tree building — find_best_split + build_tree
def find_best_split(X, y, sample_weight, feature_indices=None, criterion='gini'):
    """
    Find the optimal feature and threshold to split a node.

    Uses sorted scan (same core algorithm as sklearn's Cython implementation):
    for each candidate feature, sort samples by value, then scan left-to-right
    maintaining running weighted class counts. Each threshold evaluation is O(1)
    instead of O(n), making total cost O(n_features * n * log(n)) per node.

    Args:
        X: Features at this node (n_samples, n_features)
        y: Labels at this node (n_samples,)
        sample_weight: Per-sample weights (n_samples,)
        feature_indices: Which features to consider (None = all).
                        Random Forest passes a random sqrt(n) subset here.
        criterion: 'gini' or 'entropy'

    Returns:
        dict with 'feature', 'threshold', 'gain', or None if no valid split
    """
    n_samples = X.shape[0]

    if feature_indices is None:
        feature_indices = range(X.shape[1])

    # Parent impurity — same for all candidate splits, compute once
    imp_func = gini_impurity if criterion == 'gini' else entropy_impurity
    parent_imp = imp_func(y, sample_weight)

    best_gain = 0.0
    best_split = None

    for feat in feature_indices:
        # Sort samples by this feature's values
        sorted_idx = np.argsort(X[:, feat])
        sorted_y = y[sorted_idx]
        sorted_w = sample_weight[sorted_idx]
        sorted_vals = X[sorted_idx, feat]

        # Initialize running counts: all samples start in the right child
        n_classes = 2
        left_w = np.zeros(n_classes)
        right_w = np.zeros(n_classes)
        for c in range(n_classes):
            right_w[c] = sorted_w[sorted_y == c].sum()
        total_w = sorted_w.sum()

        # Scan left to right: move one sample at a time from right to left
        for i in range(n_samples - 1):
            c = int(sorted_y[i])
            left_w[c] += sorted_w[i]
            right_w[c] -= sorted_w[i]

            # Only evaluate at boundaries between different values
            # (splitting between identical values produces the same result)
            if sorted_vals[i] == sorted_vals[i + 1]:
                continue

            # Compute child impurities from running weight counts
            left_total = left_w.sum()
            right_total = right_w.sum()

            if criterion == 'gini':
                left_p = left_w / left_total
                right_p = right_w / right_total
                left_imp = 1.0 - np.sum(left_p ** 2)
                right_imp = 1.0 - np.sum(right_p ** 2)
            else:
                left_p = left_w / left_total
                right_p = right_w / right_total
                lp_nz = left_p[left_p > 0]
                rp_nz = right_p[right_p > 0]
                left_imp = -np.sum(lp_nz * np.log2(lp_nz))
                right_imp = -np.sum(rp_nz * np.log2(rp_nz))

            # Weighted child impurity
            child_imp = (left_total / total_w * left_imp +
                         right_total / total_w * right_imp)
            gain = parent_imp - child_imp

            if gain > best_gain:
                best_gain = gain
                best_split = {
                    'feature': feat,
                    'threshold': (sorted_vals[i] + sorted_vals[i + 1]) / 2,
                    'gain': gain
                }

    return best_split


def build_tree(X, y, sample_weight, max_depth=None, min_samples_split=2,
               feature_indices=None, criterion='gini', depth=0):
    """
    Recursively build a decision tree using greedy best-split search.

    Returns a dict-based tree compatible with tree_utils.py functions:
    - Leaf: {'value': [w0, w1], 'n_samples': n, 'impurity': imp}
    - Split: {'feature': f, 'threshold': t, 'left': {...}, 'right': {...},
              'n_samples': n, 'impurity': imp}

    Args:
        X: Features (n_samples, n_features)
        y: Labels (n_samples,)
        sample_weight: Per-sample weights (n_samples,)
        max_depth: Maximum tree depth (None = grow until pure)
        min_samples_split: Minimum samples required to attempt a split
        feature_indices: Features to consider (None = all, for RF: random subset)
        criterion: 'gini' or 'entropy'
        depth: Current depth (internal, starts at 0)

    Returns:
        dict: Tree node (recursive structure)
    """
    n_samples = len(y)
    n_classes = 2

    # Class distribution — weighted counts per class
    value = np.zeros(n_classes)
    for c in np.unique(y):
        value[int(c)] = sample_weight[y == c].sum()

    # Node impurity
    imp_func = gini_impurity if criterion == 'gini' else entropy_impurity
    impurity = imp_func(y, sample_weight)

    # Stopping conditions
    if (len(np.unique(y)) == 1 or                              # Pure node
        (max_depth is not None and depth >= max_depth) or       # Max depth reached
        n_samples < min_samples_split):                         # Too few samples
        return {'value': value, 'n_samples': n_samples, 'impurity': impurity}

    # Find best split across candidate features
    split = find_best_split(X, y, sample_weight, feature_indices, criterion)

    if split is None:
        # No valid split (e.g., all features identical at this node)
        return {'value': value, 'n_samples': n_samples, 'impurity': impurity}

    # Partition data
    left_mask = X[:, split['feature']] <= split['threshold']
    right_mask = ~left_mask

    # Recurse into children
    left_child = build_tree(
        X[left_mask], y[left_mask], sample_weight[left_mask],
        max_depth, min_samples_split, feature_indices, criterion, depth + 1
    )
    right_child = build_tree(
        X[right_mask], y[right_mask], sample_weight[right_mask],
        max_depth, min_samples_split, feature_indices, criterion, depth + 1
    )

    return {
        'feature': split['feature'],
        'threshold': split['threshold'],
        'left': left_child,
        'right': right_child,
        'value': value,
        'n_samples': n_samples,
        'impurity': impurity
    }

def get_tree_depth(node):
    """Get the maximum depth of a tree."""
    if 'feature' not in node:
        return 0
    return 1 + max(get_tree_depth(node['left']), get_tree_depth(node['right']))


def get_n_leaves(node):
    """Count the number of leaf nodes."""
    if 'feature' not in node:
        return 1
    return get_n_leaves(node['left']) + get_n_leaves(node['right'])

In [7]:
# Quick test: build a small tree (max_depth=3)
print("=" * 60)
print("TREE BUILDING — Quick Test (max_depth=3)")
print("=" * 60)

test_tree = build_tree(X_train, y_train, sample_weights, max_depth=3, criterion='gini')

print(f"  Tree depth: {get_tree_depth(test_tree)}")
print(f"  Leaves:     {get_n_leaves(test_tree)}")
print(f"  Root split: {feature_names[test_tree['feature']]} <= {test_tree['threshold']:.4f}")
print(f"  Root impurity: {test_tree['impurity']:.4f}")
print(f"  Root samples:  {test_tree['n_samples']:,}")

# Verify prediction pipeline: build_tree → flatten_tree → predict_batch
flat = flatten_tree(test_tree)
test_preds, test_proba = predict_batch(flat, X_test[:5])
print(f"\n  Predictions (first 5): {test_preds}")
print(f"  Actual (first 5):      {y_test[:5].astype(int)}")
print(f"  Proba sums to 1: {np.allclose(test_proba.sum(axis=1), 1.0)}")

TREE BUILDING — Quick Test (max_depth=3)
  Tree depth: 3
  Leaves:     8
  Root split: nr.employed <= 5087.6500
  Root impurity: 0.5000
  Root samples:  32,950

  Predictions (first 5): [1 1 1 1 1]
  Actual (first 5):      [0 0 0 0 0]
  Proba sums to 1: True
